# Геометрия по занятым сайтам: эксперимент

Полный прогон семи структур: сетка + Вороной → кластеры → симметрия → геометрия с калибровкой по занятым сайтам → координация / Li–Li / оба порядка фильтров.

Для каждого занятого симметрийного типа временно удаляется центральный Li. Его пустая позиция оценивается тем же радиусным кодом, что и кандидаты. Порог q = R_free / r_probe берётся по совместимым эталонам: одинаковые КЧ радиусной модели и состав первой сферы. Эталон может только смягчить прежний порог 90%; дополнительный допуск к его дефициту по умолчанию равен нулю.

Исходные функции поиска и радиусов не изменены. Режим включается явно; прежние результаты остаются в `voids_gallery/`, новые — в `voids_gallery_calibrated/`. Для экономии места сохраняются визуализации только этапов 06–09.

В конце — независимые контроли LGPS: поочерёдно удаляем Li каждого из четырёх исходных симметрийных типов и проверяем известное положение вакансии. Этого Li среди эталонов уже нет.

In [1]:
from pathlib import Path
from dataclasses import replace
from datetime import datetime, timezone
from contextlib import redirect_stdout, redirect_stderr
from time import perf_counter
import hashlib, importlib.metadata, io, json, sys, warnings

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "elec_voids_search/config.py").is_file())
sys.path.insert(0, str(ROOT))
from elec_voids_search.config import SearchConfig
from elec_voids_search.filtering.post_geometry_config import PostGeometryConfig
from elec_voids_search.screening_pipeline import run_screening
from elec_voids_search.results.post_geometry_exports import write_candidate_markers

EXAMPLE = ROOT / "examples/full_pipeline"
OUTPUT = EXAMPLE / "voids_gallery_calibrated"
if OUTPUT.exists():
    OUTPUT = EXAMPLE / ("voids_gallery_calibrated_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))

## Настройки

Ручные заряды и допуски видны ниже. `geometry_reference_mode="percentage"` возвращает прежнее поведение. Калибровка использует только геометрию и грубый тип окружения; подробные углы и длины связей проверяются позже.

Дефолт `geometry_reference_margin_percent=0.0`: 10% от радиуса не добавляются повторно к уже тесному занятому эталону. При отсутствии подходящего типа остаётся процентный критерий.

In [2]:
SEARCH = SearchConfig(geometry_overlap_tolerance_percent=10.0,
                      geometry_reference_mode="occupied_sites",
                      geometry_reference_shell_distance_ratio=1.25,
                      geometry_reference_margin_percent=0.0)
POST = PostGeometryConfig(paths={
    "coordination": ("coordination",),
    "li_distance": ("li_distance",),
    "coord_then_li": ("coordination", "li_distance"),
    "li_then_coord": ("li_distance", "coordination"),
})
CHARGES = {
    "LATP_sc_integer152_stoichiometric": {"Li": 1, "Ti": 4, "Al": 3, "P": 5, "O": -2},
    "LLZO_cub_ga": {"Li": 1, "La": 3, "Zr": 4, "Ga": 3, "O": -2},
    "Li10Ge(PS6)2": {"Li": 1, "Ge": 4, "P": 5, "S": -2},
    "Li3PO4": {"Li": 1, "P": 5, "O": -2},
    "LiCl": {"Li": 1, "Cl": -1},
    "LiCl_one_Li_vacancy": {"Li": 1, "Cl": -1},
    "Li3PO4_one_Li_vacancy": {"Li": 1, "P": 5, "O": -2},
}
display(pd.Series(SEARCH.to_dict(), name="Поиск и геометрия").to_frame())
display(pd.json_normalize(POST.to_dict(), sep=".").T.rename(columns={0: "После геометрии"}))

,Поиск и геометрия
grid_resolution_A,0.2
grid_search_cutoff_A,5.0
grid_symprec_A,0.1
grid_minima_window,3
grid_r_min_A,1.0
grid_empty_framework,False
grid_verbose,False
probe_element,Li
probe_oxidation_state,1
voronoi_image_range,1


,После геометрии
probe_element,Li
probe_oxidation_state,1
export_cif,False
export_rejected_cif,True
coordination.initial_cutoff_A,6.0
coordination.maximum_cutoff_A,24.0
coordination.cutoff_growth,1.5
coordination.minimum_face_area_A2,0.0
coordination.solid_angle_sum_tolerance,0.00001
coordination.significant_face_fraction,0.01


## Полный прогон

Сохраняются одна общая таблица кандидатов, радиусы, прежние нативные контроли и отдельные leave-one-out-эталоны `occupied_geometry_references.csv`. Последние — калибровочные данные, а не независимая проверка качества. CIF содержат только стадии геометрии, координации, Li–Li и пересечения.

In [3]:
OUTPUT.mkdir(parents=True, exist_ok=False)
results, summary_rows, branch_rows, run_notes = {}, [], [], {}
source_hashes = {str(p.relative_to(ROOT / "elec_voids_search")): hashlib.sha256(p.read_bytes()).hexdigest() for p in (ROOT / "elec_voids_search").rglob("*.py")}

for name, charges in CHARGES.items():
    start = perf_counter()
    path = EXAMPLE / "inputs" / f"{name}.cif"
    settings = replace(SEARCH, geometry_oxidation_states=charges)
    captured = io.StringIO()
    with redirect_stdout(captured), redirect_stderr(captured), warnings.catch_warnings(record=True) as notes:
        warnings.simplefilter("always")
        result = run_screening(path, search_config=settings, post_config=POST)
        host = Structure.from_file(path)
        search, post = result["search"], result["post_geometry"]
        branches = post["branches"]
        stages = {"01_grid": search["grid"], "02_voronoi": search["voronoi"],
                  "03_combined": search["combined"], "04_merged": search["merged"],
                  "05_symmetry": search["nonequivalent"], "06_geometry": search["geometry"],
                  "07_coordination": branches["coordination"]["passed"],
                  "08_li_distance": branches["li_distance"]["passed"],
                  "09_both": branches["coord_then_li"]["passed"]}
        folder = OUTPUT / name
        (folder / "cif").mkdir(parents=True, exist_ok=False)
        for stage, table in stages.items():
            if stage[:2] not in {"06", "07", "08", "09"}:
                continue
            write_candidate_markers(host, table, folder / "cif" / f"{stage}.cif")
        # Одна таблица вместо повторных выгрузок координат для каждого этапа.
        candidates = search["combined"].drop(columns=["merge_member_candidate_ids", "merge_source_methods"], errors="ignore").copy()
        for stage, table in stages.items():
            candidates[f"in_{stage[3:]}"] = candidates.candidate_id.isin(table.candidate_id)
        for table in (search["geometry_evaluated"], post["candidates"]):
            fields = ["candidate_id"] + [c for c in table if c not in candidates.columns]
            candidates = candidates.merge(table[fields], on="candidate_id", how="left", validate="one_to_one")
        candidates.to_csv(folder / "candidates.csv", index=False)
        search["occupied_geometry_references"].to_csv(folder / "occupied_geometry_references.csv", index=False)
        search["site_radii"].to_csv(folder / "site_radii.csv", index=False)
        search["occupied_probe_controls"].to_csv(folder / "occupied_geometry_controls.csv", index=False)
        if "occupied_reference_descriptors" in post:
            post["occupied_reference_descriptors"].to_csv(folder / "occupied_descriptors.csv", index=False)
        rows = post["summary"].copy()
        rows.insert(0, "structure", name)
        branch_rows.extend(rows.to_dict("records"))
        row = {"Соединение": name, "Сетка": len(search["grid"]), "Вороной": len(search["voronoi"]),
               "Объединено": len(search["combined"]), "Кластеры": len(search["merged"]),
               "Симметрия": len(search["nonequivalent"]), "Геометрия": len(search["geometry"]),
               "Координация": len(branches["coordination"]["passed"]),
               "Li–Li": len(branches["li_distance"]["passed"]),
               "Коорд. → Li–Li": len(branches["coord_then_li"]["passed"]),
               "Li–Li → коорд.": len(branches["li_then_coord"]["passed"])}
        summary_rows.append(row)
    results[name] = result
    run_notes[name] = {"seconds": round(perf_counter()-start, 2), "settings": settings.to_dict(),
                       "input_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                       "warnings": sorted(set(str(w.message) for w in notes))}
    print(f"{name}: геометрия {row['Геометрия']}, координация {row['Координация']}, Li–Li {row['Li–Li']}, вместе {row['Коорд. → Li–Li']}", flush=True)

summary = pd.DataFrame(summary_rows)
branch_steps = pd.DataFrame(branch_rows)
summary.to_csv(OUTPUT / "summary.csv", index=False)
branch_steps.to_csv(OUTPUT / "branch_steps.csv", index=False)
metadata = {"created_utc": datetime.now(timezone.utc).isoformat(), "search_source_sha256": source_hashes,
            "post_geometry_settings": POST.to_dict(), "structures": run_notes,
            "versions": {name: importlib.metadata.version(name) for name in ("numpy", "scipy", "pandas", "pymatgen", "gridmlip")}}
(OUTPUT / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + "\n")
print(f"Сохранено: {OUTPUT.relative_to(ROOT)}")

LATP_sc_integer152_stoichiometric: геометрия 46, координация 3, Li–Li 3, вместе 3
LLZO_cub_ga: геометрия 66, координация 40, Li–Li 17, вместе 1
Li10Ge(PS6)2: геометрия 5, координация 0, Li–Li 3, вместе 0
Li3PO4: геометрия 0, координация 0, Li–Li 0, вместе 0
LiCl: геометрия 1, координация 0, Li–Li 0, вместе 0
LiCl_one_Li_vacancy: геометрия 3, координация 1, Li–Li 1, вместе 1
Li3PO4_one_Li_vacancy: геометрия 4, координация 1, Li–Li 1, вместе 1
Сохранено: examples/full_pipeline/voids_gallery_calibrated


## Итоговые количества

«Координация» и «Li–Li» считаются независимо от геометрического входа. Последние столбцы — последовательное применение обоих фильтров. Таблицы ниже показывают каждый шаг последовательностей отдельно, включая отклонённые и неопределённые результаты.

In [4]:
display(summary)
for path, title in [("coordination", "Координация"), ("li_distance", "Li–Li"),
                    ("coord_then_li", "Координация → Li–Li"), ("li_then_coord", "Li–Li → координация")]:
    display(Markdown(f"**{title}**"))
    display(branch_steps.loc[branch_steps.path.eq(path)].drop(columns="path").reset_index(drop=True))

,Соединение,Сетка,Вороной,Объединено,Кластеры,Симметрия,Геометрия,Координация,Li–Li,Коорд. → Li–Li,Li–Li → коорд.
0,LATP_sc_integer152_stoichiometric,278,936,1214,170,88,46,3,3,3,3
1,LLZO_cub_ga,519,1271,1790,320,320,66,40,17,1,1
2,Li10Ge(PS6)2,76,330,406,136,25,5,0,3,0,0
3,Li3PO4,24,116,140,14,5,0,0,0,0,0
4,LiCl,16,10,26,6,3,1,0,0,0,0
5,LiCl_one_Li_vacancy,19,11,30,8,6,3,1,1,1,1
6,Li3PO4_one_Li_vacancy,26,104,130,17,13,4,1,1,1,1


**Координация**

,structure,step,filter,n_input,n_passed,n_rejected,n_indeterminate
0,LATP_sc_integer152_stoichiometric,1,coordination,46,3,43,0
1,LLZO_cub_ga,1,coordination,66,40,26,0
2,Li10Ge(PS6)2,1,coordination,5,0,5,0
3,Li3PO4,1,coordination,0,0,0,0
4,LiCl,1,coordination,1,0,1,0
5,LiCl_one_Li_vacancy,1,coordination,3,1,2,0
6,Li3PO4_one_Li_vacancy,1,coordination,4,1,3,0


**Li–Li**

,structure,step,filter,n_input,n_passed,n_rejected,n_indeterminate
0,LATP_sc_integer152_stoichiometric,1,li_distance,46,3,43,0
1,LLZO_cub_ga,1,li_distance,66,17,49,0
2,Li10Ge(PS6)2,1,li_distance,5,3,2,0
3,Li3PO4,1,li_distance,0,0,0,0
4,LiCl,1,li_distance,1,0,1,0
5,LiCl_one_Li_vacancy,1,li_distance,3,1,2,0
6,Li3PO4_one_Li_vacancy,1,li_distance,4,1,3,0


**Координация → Li–Li**

,structure,step,filter,n_input,n_passed,n_rejected,n_indeterminate
0,LATP_sc_integer152_stoichiometric,1,coordination,46,3,43,0
1,LATP_sc_integer152_stoichiometric,2,li_distance,3,3,0,0
2,LLZO_cub_ga,1,coordination,66,40,26,0
3,LLZO_cub_ga,2,li_distance,40,1,39,0
4,Li10Ge(PS6)2,1,coordination,5,0,5,0
5,Li10Ge(PS6)2,2,li_distance,0,0,0,0
6,Li3PO4,1,coordination,0,0,0,0
7,Li3PO4,2,li_distance,0,0,0,0
8,LiCl,1,coordination,1,0,1,0
9,LiCl,2,li_distance,0,0,0,0


**Li–Li → координация**

,structure,step,filter,n_input,n_passed,n_rejected,n_indeterminate
0,LATP_sc_integer152_stoichiometric,1,li_distance,46,3,43,0
1,LATP_sc_integer152_stoichiometric,2,coordination,3,3,0,0
2,LLZO_cub_ga,1,li_distance,66,17,49,0
3,LLZO_cub_ga,2,coordination,17,1,16,0
4,Li10Ge(PS6)2,1,li_distance,5,3,2,0
5,Li10Ge(PS6)2,2,coordination,3,0,3,0
6,Li3PO4,1,li_distance,0,0,0,0
7,Li3PO4,2,coordination,0,0,0,0
8,LiCl,1,li_distance,1,0,1,0
9,LiCl,2,coordination,0,0,0,0


## Контроль известных пустых позиций

Для двух искусственных вакансий и шести LATP-маркеров приводится минимальное периодическое расстояние до ближайшего кандидата с учётом симметрийных образов эталона. Условие восстановления позиции — ≤ 0,25 Å; это диагностический допуск, он не участвует в отборе.

In [5]:
targets = json.loads((EXAMPLE / "inputs/vacancy_targets.json").read_text())
latp = pd.read_csv(EXAMPLE / "inputs/LATP_reference_sites.csv")
checks = [(name, "removed_Li", point) for name, point in targets.items()]
checks += [("LATP_sc_integer152_stoichiometric", f"marker_{int(row.source_atom_number_1based)}",
            [row.frac_x, row.frac_y, row.frac_z]) for _, row in latp.loc[latp.resolved_state.eq("intentionally_empty")].iterrows()]
recovery_rows = []
for name, target, point in checks:
    host = Structure.from_file(EXAMPLE / "inputs" / f"{name}.cif")
    search, post = results[name]["search"], results[name]["post_geometry"]
    operations = SpacegroupAnalyzer(host, symprec=SEARCH.symmetry_symprec_A,
                                   angle_tolerance=SEARCH.symmetry_angle_tolerance_deg).get_symmetry_operations()
    orbit = [operation.operate(point) % 1 for operation in operations]
    stages = {"Сетка": search["grid"], "Вороной": search["voronoi"], "Объединено": search["combined"],
              "Кластеры": search["merged"], "Симметрия": search["nonequivalent"], "Геометрия": search["geometry"]}
    stages.update({key: branch["passed"] for key, branch in post["branches"].items()})
    for stage, table in stages.items():
        error = float(host.lattice.get_all_distances(orbit, table[["frac_x", "frac_y", "frac_z"]].to_numpy(dtype=float)).min()) if len(table) else np.nan
        recovery_rows.append({"structure": name, "target": target, "stage": stage,
                              "error_A": error, "recovered": bool(np.isfinite(error) and error <= .25)})
recovery = pd.DataFrame(recovery_rows)
recovery.to_csv(OUTPUT / "reference_recovery.csv", index=False)
display(recovery.pivot(index=["structure", "target"], columns="stage", values="recovered"))
display(recovery.loc[recovery.stage.isin(["Геометрия", "coordination", "li_distance", "coord_then_li"])].reset_index(drop=True))

/Users/irinabrodyagina/Desktop/electrolyties/elec_venv/lib/python3.12/site-packages/pymatgen/core/structure.py:3117: UserWarning: Issues encountered while parsing CIF: 6 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


stage                                         coord_then_li  ...  Симметрия
structure                         target                     ...           
LATP_sc_integer152_stoichiometric marker_146           True  ...       True
                                  marker_147           True  ...       True
                                  marker_148           True  ...       True
                                  marker_149           True  ...       True
                                  marker_150           True  ...       True
                                  marker_152           True  ...       True
Li3PO4_one_Li_vacancy             removed_Li           True  ...       True
LiCl_one_Li_vacancy               removed_Li           True  ...       True

[8 rows x 10 columns]

,structure,target,stage,error_A,recovered
0,LiCl_one_Li_vacancy,removed_Li,Геометрия,1.655591e-04,True
1,LiCl_one_Li_vacancy,removed_Li,coordination,1.655591e-04,True
2,LiCl_one_Li_vacancy,removed_Li,li_distance,1.655591e-04,True
3,LiCl_one_Li_vacancy,removed_Li,coord_then_li,1.655591e-04,True
4,Li3PO4_one_Li_vacancy,removed_Li,Геометрия,4.215082e-02,True
5,Li3PO4_one_Li_vacancy,removed_Li,coordination,4.215082e-02,True
6,Li3PO4_one_Li_vacancy,removed_Li,li_distance,4.215082e-02,True
7,Li3PO4_one_Li_vacancy,removed_Li,coord_then_li,4.215082e-02,True
8,LATP_sc_integer152_stoichiometric,marker_146,Геометрия,8.078836e-08,True
9,LATP_sc_integer152_stoichiometric,marker_146,coordination,8.078836e-08,True


## Сравнение с процентным допуском

Ранние количества должны совпадать с предыдущим полным прогоном. Колонка «Возвращено геометрией» показывает кандидатов, отвергавшихся при 10%, но принятых по совместимому занятому эталону. Она не означает подтверждённые позиции внедрения.

In [6]:
baseline = pd.read_csv(EXAMPLE / "voids_gallery/summary.csv")
comparison = []
for _, row in summary.iterrows():
    name = row["Соединение"]
    old = baseline.set_index("Соединение").loc[name]
    for column in ["Сетка", "Вороной", "Объединено", "Кластеры", "Симметрия"]:
        assert row[column] == old[column], (name, column)
    search = results[name]["search"]
    evaluated = search["geometry_evaluated"]
    references = search["occupied_geometry_references"]
    rescued = evaluated.geometry_status.eq("passed") & evaluated.geometry_uncalibrated_status.eq("rejected")
    assert set(evaluated.loc[evaluated.geometry_uncalibrated_status.eq("passed"), "candidate_id"]) <= set(search["geometry"].candidate_id)
    comparison.append({"Соединение": name, "Геометрия: 10%": int(old["Геометрия"]),
                       "Геометрия: эталоны": int(row["Геометрия"]), "Возвращено геометрией": int(rescued.sum()),
                       "Координация: было": int(old["Координация"]), "Координация: стало": int(row["Координация"]),
                       "Оба: было": int(old["Коорд. → Li–Li"]), "Оба: стало": int(row["Коорд. → Li–Li"]),
                       "Типов эталонов": len(references),
                       "Калибровочные позиции прошли": int(references.loc[references.calibrated_status.eq("passed"), "multiplicity_in_cell"].sum())})
comparison = pd.DataFrame(comparison)
comparison.to_csv(OUTPUT / "geometry_comparison.csv", index=False)
display(comparison)

,Соединение,Геометрия: 10%,Геометрия: эталоны,Возвращено геометрией,Координация: было,Координация: стало,Оба: было,Оба: стало,Типов эталонов,Калибровочные позиции прошли
0,LATP_sc_integer152_stoichiometric,46,46,0,3,3,3,3,5,10
1,LLZO_cub_ga,35,66,31,16,40,1,1,50,50
2,Li10Ge(PS6)2,5,5,0,0,0,0,0,4,20
3,Li3PO4,0,0,0,0,0,0,0,2,6
4,LiCl,1,1,0,0,0,0,0,1,2
5,LiCl_one_Li_vacancy,3,3,0,1,1,1,1,1,1
6,Li3PO4_one_Li_vacancy,4,4,0,1,1,1,1,3,5


## LGPS: независимые искусственные вакансии

Проверяем точные исходные координаты удалённого Li без генерации сетки. В эталонах остаются только другие занятые позиции. Координация проверяется также для известной позиции независимо от решения геометрии, чтобы локализовать возможный отказ.

In [7]:
from elec_voids_search.geometry.geometry import screen_geometry
from elec_voids_search.filtering.weighted_voronoi import WeightedVoronoiMatcher
from elec_voids_search.symmetry.occupied_sites import get_nonequivalent_occupied_sites

name = "Li10Ge(PS6)2"
original = Structure.from_file(EXAMPLE / "inputs" / f"{name}.cif")
settings = replace(SEARCH, geometry_oxidation_states=CHARGES[name])
analyzer = SpacegroupAnalyzer(original, symprec=SEARCH.symmetry_symprec_A,
                             angle_tolerance=SEARCH.symmetry_angle_tolerance_deg)
groups = get_nonequivalent_occupied_sites(original, analyzer, SEARCH.probe_element)
heldout_rows = []
for _, group in groups.iterrows():
    index = int(group.atom_indices[0])
    point = original[index].frac_coords.copy()
    vacancy = original.copy()
    vacancy.remove_sites([index])
    candidate = pd.DataFrame([dict(candidate_id=0, symmetry_group_id=0, source_method="known_vacancy",
                                  frac_x=point[0], frac_y=point[1], frac_z=point[2])])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        geometry = screen_geometry(vacancy, candidate, settings)
        control = geometry["geometry_evaluated"].iloc[0]
        matcher = WeightedVoronoiMatcher(vacancy, CHARGES[name], config=POST.coordination)
        coordinated = matcher.screen(candidate).iloc[0]
    heldout_rows.append({"removed_site_index": index, "source_multiplicity": int(group.multiplicity_in_cell),
                        "probe_coordination": control.probe_coordination,
                        "shell": control.geometry_reference_composition,
                        "free_radius_ratio": control.geometry_free_radius_ratio,
                        "threshold_ratio": control.geometry_threshold_ratio,
                        "reference_status": control.geometry_reference_status,
                        "percentage_status": control.geometry_uncalibrated_status,
                        "calibrated_geometry_status": control.geometry_status,
                        "coordination_status": coordinated.coordination_status})
    print(f"Удалён Li #{index}: геометрия {control.geometry_status}, координация {coordinated.coordination_status}", flush=True)
heldout = pd.DataFrame(heldout_rows)
heldout.to_csv(OUTPUT / "lgps_heldout_vacancies.csv", index=False)
display(heldout)

Удалён Li #0: геометрия passed, координация passed
Удалён Li #8: геометрия passed, координация passed
Удалён Li #10: геометрия passed, координация passed
Удалён Li #12: геометрия passed, координация passed


,removed_site_index,source_multiplicity,probe_coordination,shell,free_radius_ratio,threshold_ratio,reference_status,percentage_status,calibrated_geometry_status,coordination_status
0,0,8,6,{'S': 4},0.743765,0.743765,matched_type,rejected,passed,passed
1,8,2,6,{'S': 6},1.000734,0.900000,matched_type,passed,passed,passed
2,10,2,6,{'S': 6},1.000921,0.900000,matched_type,passed,passed,passed
3,12,8,5,{'S': 5},0.821243,0.821243,matched_type,rejected,passed,passed


## Итог эксперимента

Все четыре независимые вакансии LGPS прошли геометрию и координацию. Два типа, представляющие 16 исходных Li, ранее отсеивались процентным критерием, а теперь согласованы с оставшимися занятыми эталонами.

Для исходного LGPS дополнительных совпадений не появилось: геометрия 5 → 5, координация 0. В LLZO геометрия 35 → 66, координация 16 → 40; дополнительные точки отвергнуты Li–Li, поэтому пересечение осталось равным 1. Известные LATP-позиции и обе прежние искусственные вакансии сохранены.

Режим экспериментальный и включается явно. Проверки: 18 модульных тестов; прежний процентный критерий воспроизведён для всех семи структур; ранние этапы и координаты сохранены; проверены 56 файлов визуализации. Подробности в `voids_gallery_calibrated/README.md`.